# 🚀 TaleForge: 1-Click Free GPU LoRA Training
### Train your own private AI model on your Bengali stories using Google Colab's Free T4 GPU (15 GB VRAM).

**Instructions:**
1. In the menu, click **Runtime > Change runtime type > T4 GPU**.
2. Run the cells step-by-step (Shift + Enter).
3. Upload your `train.jsonl` exported from TaleForge (`data/datasets/train.jsonl`).
4. Download the trained LoRA adapter weights (`taleforge-lora.zip`) and place them in `models/adapters/` in your TaleForge repository!

In [ ]:
# Step 1: Verify GPU
!nvidia-smi

In [ ]:
# Step 2: Install ML Libraries
!pip install -q -U torch transformers peft datasets accelerate bitsandbytes trl

In [ ]:
# Step 3: Upload your TaleForge train.jsonl dataset
from google.colab import files
import os

print("Upload your 'train.jsonl' from TaleForge/data/datasets/:")
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"Loaded dataset: {dataset_filename}")

In [ ]:
# Step 4: Configure Base Model & Quantization
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# You can use 1.5B (fast) or 7B (deeper creative quality)
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "taleforge-lora"

print(f"Loading {BASE_MODEL} in 4-bit precision...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# Apply LoRA (Low-Rank Adaptation)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Format Dataset and Train
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

raw_dataset = load_dataset("json", data_files={"train": dataset_filename})["train"]

def tokenize_format(example):
    messages = example.get("messages")
    if messages and hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        inst = example.get("instruction", "")
        out = example.get("output", "")
        text = f"User: {inst}\nAssistant: {out}"
    tokenized = tokenizer(text, truncation=True, max_length=1024, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_ds = raw_dataset.map(tokenize_format, remove_columns=raw_dataset.column_names)

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
)

print("Starting LoRA training...")
trainer.train()

# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter successfully saved to {OUTPUT_DIR}!")

In [ ]:
# Step 6: Test Generation with your newly trained AI Model!
prompt = "একটি বৃষ্টির রাতের রোমান্টিক গল্প রচনা করো"
messages = [
    {"role": "system", "content": "You are TaleForge AI, an expert literary novelist specializing in Bengali literature."},
    {"role": "user", "content": prompt}
]
text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text_input], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.1,
)

generated_story = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Generated Story from Your Trained Model ---\n")
print(generated_story)

In [ ]:
# Step 7: Download your trained LoRA adapter to your computer
import shutil
shutil.make_archive("taleforge-lora", 'zip', OUTPUT_DIR)
files.download("taleforge-lora.zip")
print("Extract the zip contents into TaleForge/models/adapters/taleforge-lora in your project!")